# 03 Gold Tables (A): Order flow

**Owner:** Person A
**Builds:** `gold.dim_customers`, `gold.fact_orders`, `gold.fact_order_items`, `gold.mart_customer_rfm`
**Rule:** joins happen only in gold, never in silver. Aggregate one-to-many tables to one row per `order_id` BEFORE joining.

## Build order
1. `dim_customers` (no dependencies)
2. `fact_order_items`
3. `fact_orders` (Person B depends on it, so tell B when it is ready)
4. `mart_customer_rfm` (needs all three above)


---

## 1. `gold.dim_customers`
- **Grain:** one row per `customer_id`.
- **Base:** `silver.customers`
- **Joins:** none
- **Columns:** `customer_id`, `customer_unique_id`, `zip_code_prefix`, `city`, `state`.
- **Note:** a real person is `customer_unique_id` (one person can have several `customer_id` values). Use `DISTINCTCOUNT(customer_unique_id)` in Power BI for customer counts.
- **Checks:** `customer_id` unique; row count equals `silver.customers`.


In [5]:
#Importing libraries
import pandas as pd
from sqlalchemy import text
import sys
from pathlib import Path

# db.py lives in notebooks/, two levels above this notebook's folder
NOTEBOOKS_DIR = Path.cwd().parent.parent
sys.path.append(str(NOTEBOOKS_DIR))

from db import get_engine
engine = get_engine()



In [8]:
"""SELECT sorğusu: nəticəni DataFrame kimi qaytarır."""
def q(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

"""CREATE/DROP/INSERT: icra edir və commit edir."""
def run(sql):
    with engine.begin() as conn:
        conn.execute(text(sql))

In [10]:
from sqlalchemy import text

# Rebuild gold.dim_customers (safe to re-run)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.dim_customers"))
    conn.execute(text("""
        SELECT
            customer_id,
            customer_unique_id,
            customer_zip_code_prefix,
            customer_city,
            customer_state
        INTO gold.dim_customers
        FROM silver.customers
    """))

#### Checking after creating tables in gold

In [12]:
# 1. Row count must equal silver
q("""
SELECT
    (SELECT COUNT(*) FROM silver.customers)      AS silver_rows,
    (SELECT COUNT(*) FROM gold.dim_customers)    AS gold_rows
""")


,silver_rows,gold_rows
0,99441,99441


In [13]:
# 2. customer_id must be unique and not null (both numbers must match)
q("""
SELECT COUNT(*)                      AS total_rows,
       COUNT(DISTINCT customer_id)   AS distinct_ids,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_ids
FROM gold.dim_customers
""")

,total_rows,distinct_ids,null_ids
0,99441,99441,0


In [14]:
# 3. Real customers vs customer_id values (expected: fewer unique people)
q("""
SELECT COUNT(DISTINCT customer_unique_id) AS unique_customers,
       COUNT(*)                           AS customer_ids
FROM gold.dim_customers
""")

,unique_customers,customer_ids
0,96096,99441


## 2. `gold.fact_orders`
- **Grain:** one row per order.
- **Base:** `silver.orders`
- **Joins:**

| Join | Type | Aggregate first (one row per `order_id`) | Columns produced |
|---|---|---|---|
| `silver.order_payments` | `LEFT JOIN` on `order_id` | `SUM(payment_value)`, `MAX(payment_installments)`, main payment type (the type with the highest value) | `total_payment`, `payment_installments_max`, `main_payment_type` |

- **Columns:** `order_id`, `customer_id`, `order_status`, `order_date` (DATE), `approved_date`, `delivered_carrier_date`, `delivered_customer_date`, `estimated_delivery_date`, `delivery_days`, `delay_days`, `is_late` (BIT), `total_payment`, `payment_installments_max`, `main_payment_type`.
- **Logic:**
  - Aggregate `silver.order_payments` by `order_id` first (sum of payment value, main payment type).
  - `delivery_days` = delivered date minus purchase date; `delay_days` = delivered date minus estimated date; `is_late` = `delay_days > 0`.
  - Delivery columns stay `NULL` for orders that were not delivered (`is_late` is NULL, not 0).
- **Checks:** `order_id` unique; row count equals `silver.orders`; total payment matches the sum in `silver.order_payments` (explain small differences in the log).

In [15]:
#checking column names

q("""
SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME IN ('orders', 'order_payments')
ORDER BY TABLE_NAME, ORDINAL_POSITION
""")

,TABLE_NAME,COLUMN_NAME,DATA_TYPE
0,order_payments,order_id,varchar
1,order_payments,payment_sequential,smallint
2,order_payments,payment_type,varchar
3,order_payments,payment_installments,smallint
4,order_payments,payment_value,decimal
5,orders,order_id,varchar
6,orders,customer_id,varchar
7,orders,order_status,varchar
8,orders,order_purchase_timestamp,datetime2
9,orders,order_approved_at,datetime2


In [16]:
# Step 1: one row per order with total payment and max installments
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_pay"))
    conn.execute(text("""
        SELECT order_id,
               SUM(payment_value)        AS total_payment,
               MAX(payment_installments) AS payment_installments_max
        INTO gold._tmp_pay
        FROM silver.order_payments
        GROUP BY order_id
    """))

q("SELECT COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders FROM gold._tmp_pay")

,rows_,orders
0,99440,99440


In [17]:
# Step 2: main payment type = type with the highest value in the order
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_main_type"))
    conn.execute(text("""
        WITH t AS (
            SELECT order_id, payment_type,
                   ROW_NUMBER() OVER (
                       PARTITION BY order_id
                       ORDER BY SUM(payment_value) DESC, payment_type
                   ) AS rn
            FROM silver.order_payments
            GROUP BY order_id, payment_type
        )
        SELECT order_id, payment_type AS main_payment_type
        INTO gold._tmp_main_type
        FROM t
        WHERE rn = 1
    """))

q("SELECT COUNT(*) AS rows_, COUNT(DISTINCT order_id) AS orders FROM gold._tmp_main_type")

,rows_,orders
0,99440,99440


In [28]:
# Step 3: orders with delivery_days and delay_days (NULL if not delivered)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold._tmp_base"))
    conn.execute(text("""
        SELECT order_id,
               customer_id,
               order_status,
               CAST(order_purchase_timestamp AS DATE) AS order_date,
               order_approved_at                      AS approved_date,
               order_delivered_carrier_date           AS delivered_carrier_date,
               order_delivered_customer_date          AS delivered_customer_date,
               order_estimated_delivery_date          AS estimated_delivery_date,
               CASE WHEN order_delivered_customer_date IS NOT NULL
                    THEN DATEDIFF(day, CAST(order_purchase_timestamp AS DATE),
                                       CAST(order_delivered_customer_date AS DATE)) END AS delivery_days,
               CASE WHEN order_delivered_customer_date IS NOT NULL
                         AND order_estimated_delivery_date IS NOT NULL
                    THEN DATEDIFF(day, CAST(order_estimated_delivery_date AS DATE),
                                       CAST(order_delivered_customer_date AS DATE)) END AS delay_days
        INTO gold._tmp_base
        FROM silver.orders
    """))

q("SELECT COUNT(*) AS rows_ FROM gold._tmp_base")

,rows_
0,99441


In [29]:
# Step 4: final table = base + payment total + main payment type
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS gold.fact_orders"))
    conn.execute(text("""
        SELECT b.order_id,
               b.customer_id,
               b.order_status,
               b.order_date,
               b.approved_date,
               b.delivered_carrier_date,
               b.delivered_customer_date,
               b.estimated_delivery_date,
               b.delivery_days,
               b.delay_days,
               -- NULL (not 0) when delay_days is unknown
               CAST(CASE WHEN b.delay_days IS NULL THEN NULL
                         WHEN b.delay_days > 0 THEN 1
                         ELSE 0 END AS BIT)  AS is_late,
               p.total_payment,
               p.payment_installments_max,
               m.main_payment_type
        INTO gold.fact_orders
        FROM gold._tmp_base b
        LEFT JOIN gold._tmp_pay p       ON p.order_id = b.order_id
        LEFT JOIN gold._tmp_main_type m ON m.order_id = b.order_id
    """))

q("""
SELECT (SELECT COUNT(*) FROM silver.orders)    AS silver_rows,
       (SELECT COUNT(*) FROM gold.fact_orders) AS gold_rows
""")

,silver_rows,gold_rows
0,99441,99441


In [30]:
# Step 5: drop the staging tables
with engine.begin() as conn:
    for t in ["_tmp_pay", "_tmp_main_type", "_tmp_base"]:
        conn.execute(text(f"DROP TABLE IF EXISTS gold.{t}"))

## 3. `gold.fact_order_items`
- **Grain:** one row per order item (`order_id` + `order_item_id`).
- **Base:** `silver.order_items`
- **Joins:**

| Join | Type | Why | Columns produced |
|---|---|---|---|
| `silver.orders` | `LEFT JOIN` on `order_id` | Orders is one row per order, so no duplication | `customer_id`, `order_date`, `order_status` |

- **Do NOT join** products, sellers or payments. Keep `product_id` and `seller_id` as keys; Power BI relates them to the dimensions.
- **Columns:** `order_id`, `order_item_id`, `product_id`, `seller_id`, `customer_id`, `order_date`, `order_status`, `price`, `freight_value`, `item_total` (price + freight).
- **Note:** keep payments out of this table. Order-level amounts live in `fact_orders`, otherwise they repeat on every item and inflate totals.
- **Checks:** row count equals `silver.order_items`; `SUM(price)` equals the silver total.



## 4. `gold.mart_customer_rfm`
- **Grain:** one row per `customer_unique_id`.
- **Base:** `gold.fact_orders`
- **Joins:**

| Join | Type | Aggregate first | Purpose |
|---|---|---|---|
| `gold.dim_customers` on `customer_id` | `JOIN` | none | maps each order to `customer_unique_id` |
| `gold.fact_order_items` on `order_id` | `LEFT JOIN` | `SUM(price)` per `order_id` | order value for `monetary` |

  Aggregate items to one row per order BEFORE joining to `fact_orders`, then group by `customer_unique_id`.
- **Columns:** `customer_unique_id`, `recency_days`, `frequency` (distinct orders), `monetary` (sum of item prices), `r_score`, `f_score`, `m_score`, `segment`.
- **Logic:** reference date is the day after the last order in the data (not today). Scores 1-5 with `NTILE`/`qcut`; most customers order only once, so frequency scoring needs a simple rule (for example 1 order, 2 orders, 3+ orders). Document the segment rules.
- **Why a mart:** segmentation is awkward in DAX, so it is calculated once here.
- **Checks:** `customer_unique_id` unique; row count equals `COUNT(DISTINCT customer_unique_id)` in `dim_customers`; all scores are between 1 and 5; `SUM(monetary)` is close to `SUM(price)` in `fact_order_items` (the difference comes from excluded statuses, if any).

## Validation (run at the end)
- Row counts: gold vs silver for every table above.
- No duplicate keys.
- Orphan check: every `fact_orders.customer_id` and `fact_order_items.customer_id` exists in `dim_customers`.